# Week 15 Graded Mini Project
## Domain Support Assistant — E-Commerce (RAG Conversational Chatbot)

This notebook implements a **domain-specific Retrieval-Augmented Generation (RAG) conversational assistant** for the **E-Commerce** domain, using the RAG-based HR Support Chatbot from class as the reference architecture.

**Domain documents used (real, publicly available Amazon.in Customer Service pages, saved as PDF in `documents/`):**
- `Amazon_in_Returns_Policy_-_Amazon_Customer_Service.pdf`
- `About_Amazon_s_Shipping_and_Delivery_services_-_Amazon_Customer_Service.pdf`
- `Frequently_Asked_Questions_about_Warranty_-_Amazon_Customer_Service.pdf`
- `Payment_Issues_-_Amazon_Customer_Service.pdf`

Source links are listed in `README.md`.

The notebook is organized into the four required sections:
- **A) Document Ingestion**
- **B) Retrieval-Augmented Chat**
- **C) Context Awareness (conversation memory / follow-ups)**
- **D) Safety & Accuracy (grounded refusal behavior)**


## Setup: Install Dependencies

Run this cell first. It installs everything needed for document ingestion, embeddings, the vector store, and the bonus Streamlit UI.

In [1]:
%pip install -q langchain langchain-community langchain-openai faiss-cpu streamlit pymupdf python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


**Before running the next cell:** copy `.env.example` to `.env` in this same folder and replace the placeholder with your real OpenAI API key. `.env` is already listed in `.gitignore` so it won't be committed or shared.

In [1]:
import os
from dotenv import load_dotenv

# Load OPENAI_API_KEY from a local .env file (copy .env.example to .env and fill in your key)
load_dotenv()

import langchain, langchain_community
print(langchain.__version__)
print(langchain_community.__version__)

0.2.17
0.2.19


## Section A: Document Ingestion

Load the domain documents from a local folder, split them into semantic chunks, generate OpenAI embeddings, and store them in a FAISS vector store.

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

In [3]:
DATA_FOLDER = "documents"
VECTOR_STORE = "ecommerce_vector_db"
EMBEDDING_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"
TOP_K = 4
FALLBACK_MESSAGE = "I don't have enough information in the provided documents."

# Vocareum-issued keys (starting with "voc-") are proxied through Vocareum's own
# gateway, not api.openai.com. Set OPENAI_API_BASE in .env if you're using one
# (see the comment in .env.example for where to find the correct URL).
OPENAI_API_BASE = os.environ.get("OPENAI_API_BASE")

In [4]:
# Load documents from the local folder (.pdf format)
loader = DirectoryLoader(
    path=DATA_FOLDER,
    glob="./*.pdf",
    loader_cls=PyMuPDFLoader,
    use_multithreading=True,
    show_progress=True
)
documents = loader.load()
len(documents)  # note: PyMuPDFLoader returns one Document per PDF page

100%|██████████| 4/4 [00:00<00:00,  6.46it/s]


14

In [5]:
# Split content into semantic chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=50,
    separators=["\n\n", "\n", "(?<=\\. )", " ", ""]
)
splitted_text = text_splitter.split_documents(documents)
len(splitted_text)

67

In [6]:
# Generate embeddings using OpenAI embeddings
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL, base_url=OPENAI_API_BASE)

# Store embeddings in a FAISS vector store
vectordb = FAISS.from_documents(
    documents=splitted_text,
    embedding=embeddings
)
vectordb.save_local(VECTOR_STORE)
print("Vector store built and saved to", VECTOR_STORE)

Vector store built and saved to ecommerce_vector_db


## Section B: Retrieval-Augmented Chat

Retrieve the top-k relevant chunks per query, inject them into the LLM prompt, and require the answer to stay grounded strictly in that retrieved context.

In [7]:
retriever = vectordb.as_retriever(search_kwargs={"k": TOP_K})

llm = ChatOpenAI(
    model=CHAT_MODEL,
    temperature=0,
    base_url=OPENAI_API_BASE
)

In [8]:
system_prompt = f"""You are a Domain Support Assistant for an e-commerce company.

Use:
1. The provided document context to answer factually
2. Conversation history to understand follow-up questions

Rules:
- Answer ONLY using the provided document context below
- Do NOT rely on outside knowledge or invent information not present in the context
- If the answer is not present in the context, respond exactly with:
  \"{FALLBACK_MESSAGE}\"
- Be clear, concise, and professional"""

# Prompt template with a slot for conversation history (used in Section C)
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt + "\n\nDocument Context:\n{context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{question}")
])

## Section C: Context Awareness (Conversation Memory)

The assistant keeps a running conversation history and passes it into the prompt via `MessagesPlaceholder`, so follow-up questions like *"Does this apply to electronics?"* resolve correctly without repeating the full question. Retrieval itself is performed using the current question only (a standard, reasonable simplification), while the LLM sees the full recent history for reasoning about follow-ups.

In [9]:
HISTORY_TURNS = 6  # number of recent turns retained as memory

def ask(question, session_messages):
    """
    session_messages: list of {"role": "user"/"assistant", "content": str}
    Returns the assistant's answer and appends both turns to session_messages.
    """
    # Build LangChain message history from prior turns
    history = session_messages[-HISTORY_TURNS:]
    chat_history = []
    for m in history:
        if m["role"] == "user":
            chat_history.append(HumanMessage(content=m["content"]))
        elif m["role"] == "assistant":
            chat_history.append(AIMessage(content=m["content"]))

    # Retrieve context using the current question
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)

    # Invoke the LLM with grounded context + memory
    response = llm.invoke(
        prompt.format_messages(
            chat_history=chat_history,
            context=context,
            question=question
        )
    )

    session_messages.append({"role": "user", "content": question})
    session_messages.append({"role": "assistant", "content": response.content})
    return response.content

In [10]:
def chat():
    print("\nE-Commerce Support Chatbot (type 'exit' to quit)\n")
    session_messages = []
    while True:
        question = input("Customer: ")
        if question.lower() == "exit":
            break
        answer = ask(question, session_messages)
        print("\nBot:", answer)
        print("-" * 60)

## Section D: Safety & Accuracy

The system prompt instructs the model to answer **only** from retrieved context and to return the exact fallback message when the answer isn't present in the documents, preventing hallucination. The cells below demonstrate: (1) a grounded answer, (2) a follow-up handled via memory, and (3) a correct refusal for an out-of-scope question.

In [11]:
# Demonstration run (requires OPENAI_API_KEY to be set in the environment)
demo_session = []

print(ask("Are products purchased by international customers eligible for returns?", demo_session))
print("=" * 60)
print(ask("What about refunds for those international orders instead?", demo_session))  # follow-up via memory
print("=" * 60)
print(ask("What is your company's stock ticker symbol?", demo_session))   # out-of-scope -> should refuse

Products purchased by international customers are not eligible for returns. However, they are eligible for refunds if customers contact customer service within 5 business days from the delivery date or estimated delivery date to claim refunds.
International orders are eligible for refunds, and customers must contact customer service within 5 business days from the delivery date or estimated delivery date to claim refunds.
I don't have enough information in the provided documents.


### Expected behavior for the three demo questions

1. **"Are products purchased by international customers eligible for returns?"** → Grounded answer from the Amazon.in Returns Policy PDF: international customers are **not eligible for returns**, but orders made by international customers **are eligible for refunds**, and customers must contact customer service within **5 business days** of the delivery date or estimated delivery date to claim a refund.
2. **"What about refunds for those international orders instead?"** → Follow-up resolved using conversation memory (understands "those" refers to international orders from the prior turn); grounded in the same section of the Returns Policy PDF.
3. **"What is your company's stock ticker symbol?"** → Not present in any document → model should return:
   `"I don't have enough information in the provided documents."`

See `sample_conversation_log.txt` in the project folder for a full captured run.


## Bonus: Streamlit UI

A persistent-memory Streamlit chat UI is provided in `app.py` (run with `streamlit run app.py` after this notebook has built the FAISS index). It mirrors the `ask()` logic above with `st.session_state` for memory across turns in the browser.